In [3]:
import os
import numpy as np
import pandas as pd
import diptest

from sklearn.mixture import GaussianMixture

INPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_thickness_summary.csv"
OUTPUT_FOLDER = r"C:\Users\ishin\OneDrive\Desktop\ish\GMM_plots"

OUTPUT_CSV = os.path.join(
    OUTPUT_FOLDER,
    "GMM_all_patients_summary.csv"
)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(
        f"\nInput file not found:\n{INPUT_CSV}"
    )

df = pd.read_csv(INPUT_CSV)

required_columns = [
    "patient_id",
    "median_thickness_nm"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required columns:\n"
        + "\n".join(missing_columns)
    )

print()
print("GBM MULTIMODALITY ANALYSIS")
print("=" * 70)

print(
    f"Total membrane components : {len(df)}"
)

print(
    f"Patients found            : "
    f"{df['patient_id'].nunique()}"
)

print("=" * 70)

results = []

for patient_id, group in df.groupby("patient_id"):

    print()
    print("-" * 70)
    print(f"Processing Patient {patient_id}")
    print("-" * 70)

    thickness = (
        group["median_thickness_nm"]
        .dropna()
        .to_numpy()
    )

    n_samples = len(thickness)

    print(
        f"Membrane components : {n_samples}"
    )

    if n_samples < 5:

        print(
            "Skipped - too few membrane components"
        )

        continue


    X = thickness.reshape(-1, 1)

    # HARTIGAN'S DIP TEST
    dip_statistic, dip_pvalue = diptest.diptest(
        thickness
    )

    if dip_pvalue < 0.05:

        dip_classification = "Multimodal"

    else:

        dip_classification = "Unimodal"


    # FIT GMM MODELS
    models = {}
    bic = {}
    aic = {}

    for n_components in [1, 2, 3]:

        model = GaussianMixture(
            n_components=n_components,
            random_state=42,
            n_init=20
        )

        model.fit(X)

        models[n_components] = model

        bic[n_components] = model.bic(X)

        aic[n_components] = model.aic(X)


    best_components = min(
        bic,
        key=bic.get
    )

    best_model = models[best_components]

    # ORDER COMPONENTS BY MEAN
    means = best_model.means_.flatten()
    weights = best_model.weights_.flatten()
    order = np.argsort(means)
    means = means[order]
    weights = weights[order]

    peaks = np.pad(
        means,
        (0, 3 - len(means)),
        constant_values=np.nan
    )

    component_weights = np.pad(
        weights,
        (0, 3 - len(weights)),
        constant_values=np.nan
    )

    if best_components >= 2:

        peak_separation = (
            peaks[1] - peaks[0]
        )

    else:

        peak_separation = np.nan


    # INTERPRETATION
    if dip_classification == "Unimodal":

        if best_components == 1:

            interpretation = (
                "Unimodal distribution with "
                "single Gaussian component"
            )

        else:

            interpretation = (
                "Unimodal distribution with "
                "multiple Gaussian components"
            )

    else:

        interpretation = (
            "Multimodal distribution"
        )

    if dip_pvalue < 0.01:

        if best_components >= 3:

            evidence = "Very Strong"

        else:

            evidence = "Strong"

    elif dip_pvalue < 0.05:

        if best_components >= 2:

            evidence = "Strong"

        else:

            evidence = "Moderate"

    else:

        evidence = "Not Detected"


    print(
        f"Dip statistic       : "
        f"{dip_statistic:.5f}"
    )

    print(
        f"Dip p-value         : "
        f"{dip_pvalue:.5f}"
    )

    print(
        f"Dip classification  : "
        f"{dip_classification}"
    )

    print(
        f"Best GMM            : "
        f"{best_components} component(s)"
    )

    print(
        f"BIC                 : "
        f"{bic[best_components]:.2f}"
    )

    print(
        f"AIC                 : "
        f"{aic[best_components]:.2f}"
    )

    print(
        f"Peak positions      : "
        f"{np.round(peaks, 2)}"
    )

    print(
        f"Peak weights        : "
        f"{np.round(component_weights, 3)}"
    )

    print(
        f"Peak separation     : "
        f"{peak_separation:.2f}"
    )

    print(
        f"Interpretation      : "
        f"{interpretation}"
    )

    print(
        f"Statistical evidence: "
        f"{evidence}"
    )


    results.append({

        "Patient_ID":
            patient_id,

        "Number_of_membranes":
            n_samples,

        "Dip_statistic":
            dip_statistic,

        "Dip_p_value":
            dip_pvalue,

        "Dip_classification":
            dip_classification,

        "Best_GMM_components":
            best_components,

        "BIC":
            bic[best_components],

        "AIC":
            aic[best_components],

        "Peak_1_nm":
            peaks[0],

        "Peak_2_nm":
            peaks[1],

        "Peak_3_nm":
            peaks[2],

        "Weight_1":
            component_weights[0],

        "Weight_2":
            component_weights[1],

        "Weight_3":
            component_weights[2],

        "Peak_separation_nm":
            peak_separation,

        "GMM_interpretation":
            interpretation,

        "Evidence":
            evidence
    })

summary_df = pd.DataFrame(results)

if summary_df.empty:

    raise SystemExit(
        "No valid patients were analysed."
    )

# RANK PATIENTS
summary_df = summary_df.sort_values(

    by=[
        "Dip_p_value",
        "Best_GMM_components",
        "BIC"
    ],

    ascending=[
        True,
        False,
        True
    ]

).reset_index(drop=True)


summary_df.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(summary_df) + 1
    )
)

summary_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print()
print("GMM ANALYSIS COMPLETED")
print("=" * 70)

print(
    f"Results saved to:\n{OUTPUT_CSV}"
)

print("=" * 70)


GBM MULTIMODALITY ANALYSIS
Total membrane components : 394
Patients found            : 11

----------------------------------------------------------------------
Processing Patient 01-24
----------------------------------------------------------------------
Membrane components : 19
Dip statistic       : 0.06017
Dip p-value         : 0.85855
Dip classification  : Unimodal
Best GMM            : 3 component(s)
BIC                 : 204.32
AIC                 : 196.76
Peak positions      : [170.83 379.97 480.62]
Peak weights        : [0.842 0.053 0.105]
Peak separation     : 209.14
Interpretation      : Unimodal distribution with multiple Gaussian components
Statistical evidence: Not Detected

----------------------------------------------------------------------
Processing Patient 02-24
----------------------------------------------------------------------
Membrane components : 41
Dip statistic       : 0.03271
Dip p-value         : 0.99148
Dip classification  : Unimodal
Best GMM         